## Feature Engineering
---

In this notebook, the dataset is prepared for machine learning modeling.
The main goal is to transform raw variables into a suitable format for model training.
This includes encoding categorical variables, scaling numerical features if needed,
handling outliers, and selecting the final feature set.

In [17]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer

from source.features.column_standardization import column_standardization
from source.features.pipeline_output_to_dataframe import pipeline_output_to_dataframe

from sklearn.model_selection import train_test_split

In [18]:
data = pd.read_csv('../data/processed/working_dataset.csv')
data

,gender,age,hypertension,heart_disease,smoking_history,bmi,HbA1c_level,blood_glucose_level,diabetes
0,Female,80.0,0,1,never,25.19,6.6,140,0
1,Female,54.0,0,0,No Info,27.32,6.6,80,0
2,Male,28.0,0,0,never,27.32,5.7,158,0
3,Female,36.0,0,0,current,23.45,5.0,155,0
4,Male,76.0,1,1,current,20.14,4.8,155,0
...,...,...,...,...,...,...,...,...,...
99995,Female,80.0,0,0,No Info,27.32,6.2,90,0
99996,Female,2.0,0,0,No Info,17.37,6.5,100,0
99997,Male,66.0,0,0,former,27.83,5.7,155,0
99998,Female,24.0,0,0,never,35.42,4.0,100,0


In [19]:
# Wrap the custom column standardization function as a sklearn transformer.
column_standardizer = FunctionTransformer(column_standardization)

# Define the categorical columns that will be transformed with One-Hot Encoding.
categorical_columns = ['gender', 'smoking_history']

numeric_columns = [
    'age',
    'hypertension',
    'heart_disease',
    'bmi',
    'hba1c_level',
    'blood_glucose_level'
]

# Create a preprocessing object for feature transformation.
# One-Hot Encoding is applied to categorical columns.
# StandardScaler is applied to numerical columns.
# All other columns are kept unchanged because of remainder='passthrough'.
preprocessor = ColumnTransformer(
    transformers=[
        ('one_hot_encoder', OneHotEncoder(sparse_output=False), categorical_columns),
        ('scaler', StandardScaler(), numeric_columns)
    ],
    remainder='passthrough'
)

# Create the full preprocessing pipeline.
# Step 1: standardize column names
# Step 2: apply One-Hot Encoding
pipeline = Pipeline(
    steps=[
        ('column_standardization', column_standardizer),
        ('preprocessor', preprocessor)
    ]
)

# Fit the pipeline on the data and transform it.
# The result is a NumPy array, not a DataFrame.
data_encoded_array = pipeline.fit_transform(data)

# Create a preprocessing pipeline to automate the feature engineering process.
# The pipeline first standardizes the column names and then applies
# One-Hot Encoding to the selected categorical features.
#
# ColumnTransformer is used to transform only the categorical columns,
# while all remaining columns are kept unchanged.

In [20]:
pipeline

,steps,"[('column_standardization', ...), ('preprocessor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,func,<function col...002135C84C040>
,inverse_func,None
,validate,False
,accept_sparse,False
,check_inverse,True
,feature_names_out,None
,kw_args,None


In [21]:
# Convert the transformed NumPy array back into a pandas DataFrame.
data_encoded = pipeline_output_to_dataframe(
    pipeline=pipeline,
    transformed_array=data_encoded_array,
    original_data=data
)

data_encoded.head()

,gender_female,gender_male,gender_other,smoking_history_no info,smoking_history_current,smoking_history_ever,smoking_history_former,smoking_history_never,smoking_history_not current,scaler__age,scaler__hypertension,scaler__heart_disease,scaler__bmi,scaler__hba1c_level,scaler__blood_glucose_level,diabetes
0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.692704,-0.284439,4.936379,-0.321056,1.001706,0.047704,0.0
1,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.538006,-0.284439,-0.202578,-0.000116,1.001706,-1.426210,0.0
2,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,-0.616691,-0.284439,-0.202578,-0.000116,0.161108,0.489878,0.0
3,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,-0.261399,-0.284439,-0.202578,-0.583232,-0.492690,0.416183,0.0
4,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.515058,3.515687,4.936379,-1.081970,-0.679490,0.416183,0.0


In [22]:
# Split the processed dataset into training and validation datasets.
# 80% of the data will be used for model training.
# 20% of the data will be kept aside for final validation.
#
# stratify=data['diabetes'] keeps the same target class distribution
# in both train and validation datasets.
train_data, validation_data = train_test_split(
    data_encoded,
    test_size=0.20,
    random_state=42,
    stratify=data['diabetes']
)

# Save the training dataset to a CSV file.
train_data.to_csv('../data/processed/train_data.csv', index=False)

# Save the validation dataset to a CSV file.
validation_data.to_csv('../data/processed/validation_data.csv', index=False)